In [1]:
# Web scraping dan automatisasi
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
# from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException
# from selenium.webdriver.common.action_chains import ActionChains

# Parsing dan data handling
from bs4 import BeautifulSoup
import pandas as pd
import requests
import json
import re
import os
import time
import warnings

In [3]:
from pandas import options


class RegencyCityScraper:
    def __init__(self):
        warnings.filterwarnings("ignore", category=DeprecationWarning)
        self.options = Options()
        self.options.add_argument('--disable-blink-features=AutomationControlled')
        self.options.add_argument('--start-maximized')
        
        # Force UI language
        self.options.add_argument("--lang=en-US")

        # Force Accept-Language header
        self.options.add_experimental_option(
            "prefs",
            {"intl.accept_languages": "en,en_US"}
        )

        self.options.add_experimental_option("excludeSwitches", ["enable-automation"])
        self.options.add_experimental_option('useAutomationExtension', False)
        self.driver = webdriver.Chrome(options=self.options)
        self.wait = WebDriverWait(self.driver, 10)

    def scrollPage(self):
        """ Scroll the results paneluntil no more results of City/Region in Indonesia"""
        
        print("Scrolling to load all results...")

        try:

            table = self.wait.until(
                EC.presence_of_element_located((By.ID, "mw2g"))
            )

            self.driver.execute_script(
                "arguments[0].scrollIntoView({block: 'start'});",
                table
            )
            time.sleep(1.5)

            print("Table found. Scrolling table content...")
            print("-" * 50)

            previousCount = 0
            scrollIterations = 0

            # Infinite scroll until no new results are loaded
            while True:
                rows = table.find_elements(By.TAG_NAME, 'tr')[1:]
                currentCount = len(rows)
                print(f"Current loaded results: {currentCount}")

                self.driver.execute_script(
                    "arguments[0].scrollTop = arguments[0].scrollHeight",
                    table
                )
                time.sleep(1.5)
                scrollIterations += 1

                print(f"Scroll {scrollIterations}. Found {currentCount} results loaded.")
                
                if currentCount == previousCount:
                    print("No new results loaded. Finished scrolling.")
                    break

                previousCount = currentCount
        except NoSuchElementException as e:
            print(f"Not find scrollable element: {e}")

    def mainScraping(self, url):
        """ Extract City/Region data from the loaded page """
        self.driver.get(url)
        time.sleep(4)

        self.scrollPage()
        time.sleep(4)
        
        data = []
        
        rows = self.driver.find_elements(By.XPATH, '//tbody[@id="mw2g"]/tr')[1:]
        print("Total rows:", len(rows))

        print("-" * 14 + " Start Extracting Data " + "-" * 14)
        # Looping through each row to extract data
        for i, row in enumerate(rows, 1):
            try:
                column = row.find_elements(By.XPATH, './/td')
                
                if len(column) >= 5:
                    # Extract Regency Name
                    try:
                        regencyCityElement = column[1].find_element(By.TAG_NAME, 'a')
                        regencyCityTitle = regencyCityElement.get_attribute('title')
                    except:
                        regencyCityTitle = column[1].text.strip()

                    # Extract Province Name
                    try:
                        provinceElement = column[4].find_element(By.TAG_NAME, 'a')
                        provinceTitle = provinceElement.get_attribute('title')
                    except:
                        provinceTitle = column[4].text.strip()

                    print(f"[{i}/{len(rows)}] Get Data Province: {provinceTitle} | Regency/City: {regencyCityTitle}")

                    # Append to data list
                    data.append({
                        "Regency/City": regencyCityTitle,
                        "Province": provinceTitle
                    })

            except NoSuchElementException:
                continue
        return pd.DataFrame(data)

    def mappingProvinceToIsland(self, df):
        """
        Added Island column based on Province mapping
        1. Sumatera
        2. Jawa
        3. Kalimantan
        4. Sulawesi
        5. Nusa Tenggara & Bali
        6. Papua & Maluku
        7. Lainnya
        """

        province_to_island = {
            # Sumatera
            "Aceh": "Sumatera",
            "Sumatera Utara": "Sumatera",
            "Sumatera Barat": "Sumatera",
            "Riau": "Sumatera",
            "Jambi": "Sumatera",
            "Sumatera Selatan": "Sumatera",
            "Bengkulu": "Sumatera",
            "Lampung": "Sumatera",
            "Kepulauan Bangka Belitung": "Sumatera",
            "Kepulauan Riau": "Sumatera",

            # Jawa
            "Daerah Khusus Ibukota Jakarta": "Jawa",
            "Banten": "Jawa",
            "Jawa Barat": "Jawa",
            "Jawa Tengah": "Jawa",
            "Jawa Timur": "Jawa",
            "Daerah Istimewa Yogyakarta": "Jawa",

            # Kalimantan
            "Kalimantan Barat": "Kalimantan",
            "Kalimantan Tengah": "Kalimantan",
            "Kalimantan Selatan": "Kalimantan",
            "Kalimantan Timur": "Kalimantan",
            "Kalimantan Utara": "Kalimantan",

            # Sulawesi
            "Sulawesi Utara": "Sulawesi",
            "Sulawesi Tengah": "Sulawesi",
            "Sulawesi Selatan": "Sulawesi",
            "Sulawesi Tenggara": "Sulawesi",
            "Sulawesi Barat": "Sulawesi",
            "Gorontalo": "Sulawesi",

            # Bali & Nusa Tenggara
            "Bali": "Nusa Tenggara & Bali",
            "Nusa Tenggara Barat": "Nusa Tenggara & Bali",
            "Nusa Tenggara Timur": "Nusa Tenggara & Bali",

            # Papua & Maluku
            "Maluku": "Papua & Maluku",
            "Maluku Utara": "Papua & Maluku",
            "Papua": "Papua & Maluku",
            "Papua Barat": "Papua & Maluku",
            "Papua Barat Daya": "Papua & Maluku",
            "Papua Tengah": "Papua & Maluku",
            "Papua Pegunungan": "Papua & Maluku",
            "Papua Selatan": "Papua & Maluku",
        }

        df = df.copy()
        df["Island"] = df["Province"].map(province_to_island).fillna("Lainnya")
        return df
    
    def save_to_excel(self, df, filename):
        """ Save DataFrame to Excel file """
        df.to_excel(filename, index=False)
        print("-" * 50)
        print(f"Saved to {filename}")
        print(f"Total Regency/City: {df['Regency/City'].nunique()}")
    
    def close(self):
        """ Close the browser """
        self.driver.quit()

if __name__ == "__main__":
    scraper = RegencyCityScraper()
    url = "https://id.wikipedia.org/wiki/Kabupaten_dan_kota_di_Indonesia"
    df = scraper.mainScraping(url)
    df = scraper.mappingProvinceToIsland(df)
    scraper.save_to_excel(df, "kota_indonesia.xlsx")
    scraper.close()

Scrolling to load all results...
Table found. Scrolling table content...
--------------------------------------------------
Current loaded results: 513
Scroll 1. Found 513 results loaded.
Current loaded results: 513
Scroll 2. Found 513 results loaded.
No new results loaded. Finished scrolling.
Total rows: 513
-------------- Start Extracting Data --------------
[1/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Barat Daya
[2/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Besar
[3/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Jaya
[4/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Selatan
[5/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Singkil
[6/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Tamiang
[7/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Tengah
[8/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Tenggara
[9/513] Get Data Province: Aceh | Regency/City: Kabupaten Aceh Timur
[1